# 🏥 Fed-ISIC2019 연합학습 실험 노트북 (Colab Phase-by-Phase Runner)

팀원들이 Google Colab 환경에서 **2단계 전략(1시드 빠른 탐색 $\rightarrow$ 최종 3시드 검증)** 으로 실험을 수행하고, 결과(`Mean ± Std` 요약표 및 표준편차 음영 그래프)를 구글 드라이브에 안전하게 저장할 수 있도록 구성된 노트북입니다.

### 📋 실험 구성 및 전략 (방식 B)
- **모델**: `EfficientNet-B0` (ImageNet Pretrained, `BatchNorm` 유지, 8-Class Classifier)
- **데이터 분할 모드 (`--mode`)**:
  - `top3`: 상위 3개 주요 센터 (센터 0, 1, 2)
  - `merge6`: 전체 6개 센터(0~5)를 3개 클라이언트로 병합 `[(0,3), (1,4), (2,5)]`
- **2단계 시드 실험 전략**:
  - **Stage 1 (탐색 단계, `seed=42`)**: 단일 시드로 `Centralized`, `FedAvg`, 그리고 `FedProx`의 $\mu$ 후보값들을 빠르게 비교해 최적의 $\mu^*$를 선정합니다.
  - **Stage 2 (최종 검증 단계, `seeds=[42, 100, 2026]`)**: 이미 `seed=42`는 Stage 1에서 완료되었으므로, 최종 확정된 설정(`Centralized`, `FedAvg`, `FedProx(best_mu)`)에 대해 **나머지 2개 시드(`100, 2026`)만 추가 실행**해 3시드 `Mean ± Std` 결과를 완성합니다.
- **핵심 평가지표**:
  1. `balanced_acc`: 클래스 불균형을 고려한 균형 정확도 (체크포인트 저장 기준)
  2. `macro_f1`: 전체 8개 클래스 Macro F1-Score
  3. `macro_f1_gap` (`max_macro_f1 - min_macro_f1`): 클라이언트 간 최고-최저 Macro-F1 격차 (**병원 간 공정성 지표**)
  4. `ece`: 15개 구간(`n_bins=15`) 기준 Expected Calibration Error (**신뢰도 보정 지표**)
  5. `weight_divergence`: 글로벌 모델과 로컬 모델 간 L2 거리 ($\|w_{\text{local}} - w_{\text{global}}\|_2$)

---
## Phase 0. 코랩 GPU 확인 및 구글 드라이브 마운트
코랩 세션이 중간에 끊겨도 매 라운드 기록되는 CSV 파일과 `.pt` 가중치가 날아가지 않도록 Google Drive를 연결합니다.

In [ ]:
import os
import torch

print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device     : {torch.cuda.get_device_name(0)}")
!nvidia-smi

In [ ]:
# Google Drive 마운트 (코랩 환경에서 실행)
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
    print("로컬 환경에서 실행 중입니다.")

# 프로젝트 작업 경로 및 결과 저장 경로 설정
PROJECT_DIR = "/content/CDII" if IN_COLAB else os.getcwd()
SAVE_DIR = "/content/drive/MyDrive/CDII_results" if IN_COLAB else "./results"
os.makedirs(SAVE_DIR, exist_ok=True)
print(f"[설정 완료] 결과 저장 경로(SAVE_DIR): {SAVE_DIR}")

---
## Phase 1. 프로젝트 코드 준비, 패키지 설치 및 데이터로더 검증
GitHub 레포지토리 클론(또는 드라이브에서 복사) 후 `requirements.txt`와 `flamby`를 설치하고, 데이터셋 로더가 정상 작동하는지 확인합니다.

In [ ]:
# [선택] Git 레포지토리에서 클론하거나 이미 업로드된 프로젝트 폴더로 이동
# REPO_URL = "https://github.com/<조직_또는_계정>/<레포지토리명>.git"
# if IN_COLAB and not os.path.exists(PROJECT_DIR):
#     !git clone {REPO_URL} {PROJECT_DIR}

if os.path.exists(PROJECT_DIR):
    %cd {PROJECT_DIR}
print("현재 작업 디렉토리:", os.getcwd())
!ls -la

In [ ]:
# 필수 패키지 및 FLamby (Fed-ISIC2019 지원) 설치
!pip install -q -r requirements.txt
!pip install -q "flamby[isic2019]" || pip install -q git+https://github.com/owkin/FLamby.git

In [ ]:
# 데이터로더 및 역빈도 클래스 가중치 정상 작동 확인 (Smoke Test)
from data_utils import get_dataloaders, compute_class_weights

MODE_TEST = "top3"  # 'top3' 또는 'merge6'
client_loaders, centralized_loader, test_loader = get_dataloaders(
    mode=MODE_TEST,
    batch_size=32,
    num_workers=2,
    seed=42,
)

print(f"=== Mode: {MODE_TEST} 데이터셋 요약 ===")
for idx, loader in enumerate(client_loaders):
    w = compute_class_weights(loader, num_classes=8)
    print(f"Client {idx} | Train 샘플 수: {len(loader.dataset):5d} | Test 샘플 수: {len(test_loader.client_test_loaders[idx].dataset):4d} | 클래스 가중치: {w.numpy().round(2)}")

print(f"Centralized 통합 Train 샘플 수: {len(centralized_loader.dataset)}")
print(f"Global 통합 Test 샘플 수      : {len(test_loader.dataset)}")

---
## Phase 2. 공통 하이퍼파라미터 및 시드 설정
- **탐색 시드(`EXPLORE_SEED`)**: `42` (Stage 1에서 단일 시드로 빠르게 비교)
- **최종 검증 추가 시드(`EXTRA_SEEDS`)**: `[100, 2026]` (Stage 2에서 최종 모델에 대해서만 추가 실행하여 총 3시드 `[42, 100, 2026]` 완성)

In [ ]:
# 실험 공통 하이퍼파라미터
MODE = "top3"                  # 'top3' 또는 'merge6'
ROUNDS = 20                    # 총 통신 라운드 수
LOCAL_EPOCHS = 1               # 라운드당 클라이언트 로컬 학습 에폭 수
BATCH_SIZE = 32                # 배치 크기
LR = 1e-4                      # 학습률
NUM_WORKERS = 2                # 코랩 기본 런타임 권장값: 2

# 방식 B 시드 구성
EXPLORE_SEED = 42              # Stage 1: 단일 시드 탐색용
EXTRA_SEEDS = [100, 2026]      # Stage 2: 최종 3시드 완성용 추가 시드

print(f"[실험 설정] MODE={MODE}, ROUNDS={ROUNDS}, LOCAL_EPOCHS={LOCAL_EPOCHS}, BATCH_SIZE={BATCH_SIZE}, LR={LR}")
print(f"[시드 설정] Stage 1 탐색 시드={EXPLORE_SEED} | Stage 2 추가 시드={EXTRA_SEEDS} (총 3시드)")

---
## Phase 3. [Stage 1] 단일 시드(`seed=42`) 탐색 실험
단일 시드(`seed=42`)로 **Centralized**, **FedAvg**, 그리고 **FedProx($\mu$ 후보군)** 를 먼저 1번씩 실행하여 가장 성능이 우수한 최적의 $\mu^*$를 찾습니다.

In [ ]:
# 3-1. Centralized (seed=42)
!python main.py \
    --method centralized \
    --mode {MODE} \
    --rounds {ROUNDS} \
    --local_epochs {LOCAL_EPOCHS} \
    --batch_size {BATCH_SIZE} \
    --lr {LR} \
    --num_workers {NUM_WORKERS} \
    --seed {EXPLORE_SEED} \
    --save_dir "{SAVE_DIR}"

In [ ]:
# 3-2. FedAvg (seed=42)
!python main.py \
    --method fedavg \
    --mode {MODE} \
    --rounds {ROUNDS} \
    --local_epochs {LOCAL_EPOCHS} \
    --batch_size {BATCH_SIZE} \
    --lr {LR} \
    --mu 0.0 \
    --num_workers {NUM_WORKERS} \
    --seed {EXPLORE_SEED} \
    --save_dir "{SAVE_DIR}"

In [ ]:
# 3-3. FedProx mu 후보군 탐색 (seed=42)
# 필요에 따라 후보 리스트를 조정하세요 (예: [0.001, 0.01, 0.1] 또는 단일값 [0.01])
MU_CANDIDATES = [0.01, 0.1]

for mu_val in MU_CANDIDATES:
    print(f"\n================ [Stage 1] FedProx (mu={mu_val}, seed={EXPLORE_SEED}) ================")
    !python main.py \
        --method fedprox \
        --mode {MODE} \
        --rounds {ROUNDS} \
        --local_epochs {LOCAL_EPOCHS} \
        --batch_size {BATCH_SIZE} \
        --lr {LR} \
        --mu {mu_val} \
        --num_workers {NUM_WORKERS} \
        --seed {EXPLORE_SEED} \
        --save_dir "{SAVE_DIR}"

In [ ]:
# 3-4. Stage 1 (seed=42) 탐색 결과 비교 및 최적 mu 확인
import glob
import pandas as pd

stage1_files = sorted(glob.glob(os.path.join(SAVE_DIR, f"*_{MODE}_seed{EXPLORE_SEED}_metrics.csv")))
stage1_rows = []
for path in stage1_files:
    df = pd.read_csv(path)
    if df.empty:
        continue
    best_row = df.loc[df["balanced_acc"].idxmax()]
    stage1_rows.append({
        "File": os.path.basename(path),
        "Method": best_row["method"],
        "Mu": best_row["mu"],
        "Best Round": int(best_row["round"]),
        "Balanced Acc (↑)": best_row["balanced_acc"],
        "Macro F1 (↑)": best_row["macro_f1"],
        "Max-Min F1 Gap (↓)": best_row["macro_f1_gap"],
        "ECE (15-bin, ↓)": best_row["ece"],
    })

print(f"=== [Stage 1] Seed {EXPLORE_SEED} 탐색 결과 요약 ===")
display(pd.DataFrame(stage1_rows).round(4))

---
## Phase 4. [Stage 2] 최종 확정 설정에 대한 나머지 2개 시드(`100, 2026`) 추가 실행
Stage 1에서 이미 `seed=42`는 실행되었으므로, 최종 비교 대상인 **Centralized**, **FedAvg**, **FedProx(`BEST_MU`)** 에 대해 나머지 시드(`EXTRA_SEEDS = [100, 2026]`)만 추가로 돌려 총 3시드 실험을 완성합니다.

In [ ]:
# Stage 1 결과를 바탕으로 선정한 최적의 FedProx mu 값 지정
BEST_MU = 0.01

for s in EXTRA_SEEDS:
    print(f"\n================ [Stage 2] Centralized (seed={s}) ================")
    !python main.py \
        --method centralized \
        --mode {MODE} \
        --rounds {ROUNDS} \
        --local_epochs {LOCAL_EPOCHS} \
        --batch_size {BATCH_SIZE} \
        --lr {LR} \
        --num_workers {NUM_WORKERS} \
        --seed {s} \
        --save_dir "{SAVE_DIR}"

    print(f"\n================ [Stage 2] FedAvg (seed={s}) ================")
    !python main.py \
        --method fedavg \
        --mode {MODE} \
        --rounds {ROUNDS} \
        --local_epochs {LOCAL_EPOCHS} \
        --batch_size {BATCH_SIZE} \
        --lr {LR} \
        --mu 0.0 \
        --num_workers {NUM_WORKERS} \
        --seed {s} \
        --save_dir "{SAVE_DIR}"

    print(f"\n================ [Stage 2] FedProx (mu={BEST_MU}, seed={s}) ================")
    !python main.py \
        --method fedprox \
        --mode {MODE} \
        --rounds {ROUNDS} \
        --local_epochs {LOCAL_EPOCHS} \
        --batch_size {BATCH_SIZE} \
        --lr {LR} \
        --mu {BEST_MU} \
        --num_workers {NUM_WORKERS} \
        --seed {s} \
        --save_dir "{SAVE_DIR}"

---
## Phase 5. 3시드 통합 `Mean ± Std` 결과 요약표 및 표준편차 음영 그래프
`SAVE_DIR`에 저장된 모든 시드의 CSV 파일을 자동으로 그룹화하여 **방법론별 `평균 ± 표준편차` 요약 테이블**과 **라운드별 평균 곡선 + `± 1 Std` 음영 그래프**를 생성합니다.

In [ ]:
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

csv_files = sorted(glob.glob(os.path.join(SAVE_DIR, f"*_{MODE}_seed*_metrics.csv")))
print(f"로드된 CSV 파일 ({len(csv_files)}개):", [os.path.basename(p) for p in csv_files])

best_records = []
all_round_dfs = []

for path in csv_files:
    df = pd.read_csv(path)
    if df.empty:
        continue
    method = df["method"].iloc[0]
    mu_val = df["mu"].iloc[0] if "mu" in df.columns else 0.0
    seed_val = int(df["seed"].iloc[0]) if "seed" in df.columns else 42
    group_name = f"{method} (mu={mu_val:g})" if method == "fedprox" else method

    df["group"] = group_name
    all_round_dfs.append(df)

    # 각 시드별 최고 Balanced Accuracy 달성 라운드 추출
    best_row = df.loc[df["balanced_acc"].idxmax()]
    best_records.append({
        "Group": group_name,
        "Seed": seed_val,
        "Best Round": int(best_row["round"]),
        "balanced_acc": best_row["balanced_acc"],
        "macro_f1": best_row["macro_f1"],
        "macro_f1_gap": best_row["macro_f1_gap"],
        "ece": best_row["ece"],
        "weight_divergence": best_row["weight_divergence"],
    })

best_df = pd.DataFrame(best_records)
if not best_df.empty:
    print("\n=== 1) 개별 시드별 최고 성능 기록 ===")
    display(best_df.round(4))

    # 그룹별 Mean ± Std 요약 테이블 생성
    metrics = ["balanced_acc", "macro_f1", "macro_f1_gap", "ece", "weight_divergence"]
    agg_rows = []
    for grp, gdf in best_df.groupby("Group", sort=False):
        row = {"Method": grp, "N_Seeds": len(gdf)}
        for m in metrics:
            mean_v = gdf[m].mean()
            std_v = gdf[m].std(ddof=1) if len(gdf) > 1 else 0.0
            row[m] = f"{mean_v:.4f} ± {std_v:.4f}"
        agg_rows.append(row)

    summary_table = pd.DataFrame(agg_rows)
    summary_csv_path = os.path.join(SAVE_DIR, f"summary_mean_std_{MODE}.csv")
    summary_table.to_csv(summary_csv_path, index=False, encoding="utf-8-sig")
    print(f"\n=== 2) 다중 시드 평균 ± 표준편차 (Mean ± Std) 최종 요약표 (저장됨: {summary_csv_path}) ===")
    display(summary_table)

In [ ]:
# 라운드별 다중 시드 평균 실선 + 표준편차(±1 Std) 음영 그래프 시각화
if all_round_dfs:
    combined_df = pd.concat(all_round_dfs, ignore_index=True)
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))

    metrics_to_plot = [
        ("balanced_acc", "Global Balanced Accuracy (↑)", axes[0, 0]),
        ("macro_f1_gap", "Client Max-Min Macro-F1 Gap / Fairness (↓)", axes[0, 1]),
        ("ece", "Expected Calibration Error - 15 Bins (↓)", axes[1, 0]),
        ("weight_divergence", "Client Weight Divergence L2 Norm", axes[1, 1]),
    ]

    for col, title, ax in metrics_to_plot:
        for grp, gdf in combined_df.groupby("group", sort=False):
            stats = gdf.groupby("round")[col].agg(["mean", "std"]).reset_index()
            rounds = stats["round"].values
            mean_vals = stats["mean"].values
            std_vals = stats["std"].fillna(0.0).values

            line, = ax.plot(rounds, mean_vals, marker="o", linewidth=2, label=f"{grp} (n={gdf['seed'].nunique()})")
            if gdf["seed"].nunique() > 1:
                ax.fill_between(
                    rounds,
                    mean_vals - std_vals,
                    mean_vals + std_vals,
                    color=line.get_color(),
                    alpha=0.2,
                )

        ax.set_title(title, fontsize=12, fontweight="bold")
        ax.set_xlabel("Communication Round")
        ax.set_ylabel(col)
        ax.grid(True, linestyle="--", alpha=0.5)
        ax.legend()

    plt.tight_layout()
    plot_path = os.path.join(SAVE_DIR, f"comparison_curves_mean_std_{MODE}.png")
    plt.savefig(plot_path, dpi=200)
    plt.show()
    print(f"[저장 완료] 다중 시드 평균±표준편차 비교 그래프: {plot_path}")